# March Machine Learning Mania 2026

**Goal.** Predict, for every possible matchup in the 2026 NCAA men's and women's basketball tournaments, the probability that the team with the **lower ID** wins.

**Metric.** Brier score = mean of $(p - y)^2$, where $p$ is the predicted probability and $y$ is 1 if the lower-ID team won, 0 otherwise. Lower is better. Predicting 0.5 for every game scores 0.25.

**Approach.** Start from the simplest possible baseline (tournament seeds), then add one idea at a time and keep it only if it lowers the Brier score on games the model has not seen.

### Results so far (walk-forward backtest, 2015–2025, 1,315 games)

| Model | Men | Women | Combined |
|---|---|---|---|
| Predict 0.5 for everything | 0.250 | 0.250 | 0.250 |
| Seed difference only | 0.196 | 0.150 | 0.173 |
| **Final: men = efficiency + Elo, women = efficiency** | **0.186** | **0.138** | **0.163** |

### How to run
Run the cells top to bottom (*Runtime → Run all*). Every cell can be re-run safely.
Sections 2–8 are the step-by-step exploration. Section 9 is a self-contained **toolbox**; from Section 10 on, everything only needs Section 1 + the toolbox.

## 1. Setup and data

Put `march-machine-learning-mania-2026.zip` (downloaded from the competition's *Data* page) into a Google Drive folder called **`mania2026`**. The cell below mounts Drive and unzips it. If the zip is not found, an upload button appears instead.

In [ ]:
import os, zipfile

ZIP = 'march-machine-learning-mania-2026.zip'
DRIVE_ZIP = '/content/drive/MyDrive/mania2026/' + ZIP

if not os.path.exists('data'):
    try:
        from google.colab import drive, files            # running in Colab
        drive.mount('/content/drive')
        if os.path.exists(DRIVE_ZIP):
            src = DRIVE_ZIP
        else:
            print('Zip not found in Drive, please upload it:')
            src = list(files.upload())[0]
    except ImportError:                                   # running in local Jupyter
        src = ZIP
    zipfile.ZipFile(src).extractall('data')

print('Number of data files:', len(os.listdir('data')))

## 2. Understanding the metric: why honest probabilities win

Each game is a point $p$ on the interval $[0, 1]$; the true result sits at one of the ends (0 or 1). The Brier penalty is the **squared distance** to the correct end, so small misses cost little and confident misses cost a lot.

Suppose a team truly wins 70% of the time. If we report $p$, the **expected penalty** is

$$0.7\,(p-1)^2 + 0.3\,p^2 = (p - 0.7)^2 + 0.21.$$

This is a parabola with its minimum exactly at $p = 0.7$. Reporting 0.9 to "be bold" is strictly worse. The term $0.21 = 0.7 \times 0.3$ is irreducible randomness that no model can remove, which is why even the best entries score around 0.12, not 0.

**Takeaway:** the score rewards being *accurate*, not being *extreme*.

In [ ]:
import numpy as np
import pandas as pd

true_p = 0.7
for p in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    expected = true_p * (p - 1) ** 2 + (1 - true_p) * p ** 2
    print(f'report {p:.1f}  ->  expected penalty {expected:.3f}')

## 3. A first look at the data

We start with four men's files (the women's files have the same structure with a `W` prefix):

| File | One row is |
|---|---|
| `MRegularSeasonCompactResults` | a regular-season game: season, day, winner, winner score, loser, loser score, location |
| `MNCAATourneyCompactResults` | a tournament game: the kind of game we have to predict |
| `MNCAATourneySeeds` | a team's seed (1 = strongest, 16 = weakest in its region) |
| `MTeams` | team ID ↔ school name |

`WLoc` is the winner's location: **H** = home, **A** = away, **N** = neutral.

In [ ]:
D = 'data/'
rs    = pd.read_csv(D + 'MRegularSeasonCompactResults.csv')   # men's regular season
tc    = pd.read_csv(D + 'MNCAATourneyCompactResults.csv')     # men's tournament
seeds = pd.read_csv(D + 'MNCAATourneySeeds.csv')              # men's seeds
teams = pd.read_csv(D + 'MTeams.csv')                         # ID -> name

rs.tail()

In [ ]:
# Last row: Cal Baptist (1465) won 63-61 away at Utah Valley (1430)
teams[teams.TeamID.isin([1465, 1430])]

In [ ]:
print(rs.WLoc.value_counts())                                  # home wins far outnumber away wins
print('2025 tournament games:', len(tc[tc.Season == 2025]))    # 63 + 4 First Four games = 67

### Re-framing results as "A vs B"

The raw tables are written from the **winner's** point of view, which we never know before a game. The submission asks for the probability that the **lower-ID team** wins, so we rewrite every game in that form:

- **A** = lower team ID, **B** = higher team ID (a seating convention fixed before tip-off; IDs carry no information about strength)
- **y = 1** if A won, which is simply `WTeamID < LTeamID`

Features must only use information available before the game; the result lives only in `y`. Mixing the result into the features is called **data leakage**.

In [ ]:
tc['y'] = (tc.WTeamID < tc.LTeamID).astype(int)
print('Share of games won by A:', round(tc.y.mean(), 3))      # close to 0.5: IDs are uninformative
tc[['WTeamID', 'LTeamID', 'y']].head()

## 4. Baseline: seeds only

Seeds are the selection committee's ranking. The simplest model is a straight line:

$$p = 0.5 + k \times (\text{seed}_B - \text{seed}_A)$$

Equal seeds give 0.5; a 1-seed (A) against a 16-seed (B) gives $0.5 + 15k$.

In [ ]:
# Seed 'W01' -> 1, 'X16a' -> 16
seeds['seed'] = seeds.Seed.str[1:3].astype(int)
s = seeds[['Season', 'TeamID', 'seed']]

# Rebuild from the raw file so this cell can be re-run safely
tc = pd.read_csv(D + 'MNCAATourneyCompactResults.csv')
tc['y'] = (tc.WTeamID < tc.LTeamID).astype(int)
tc['A'] = tc[['WTeamID', 'LTeamID']].min(axis=1)
tc['B'] = tc[['WTeamID', 'LTeamID']].max(axis=1)
tc = tc.merge(s.rename(columns={'TeamID': 'A', 'seed': 'seedA'}), on=['Season', 'A'])
tc = tc.merge(s.rename(columns={'TeamID': 'B', 'seed': 'seedB'}), on=['Season', 'B'])
tc['seed_diff'] = tc.seedB - tc.seedA            # positive = A has the better seed

k = 0.03
p = 0.5 + k * tc.seed_diff
print('Brier (k = 0.03):', round(((p - tc.y) ** 2).mean(), 4), '   vs 0.25 for always-0.5')

### How to choose k

Because $p$ is linear in $k$ and the penalty is squared, the Brier score as a function of $k$ is a **parabola**. Three ways to find its bottom:

1. **Grid search:** try evenly spaced values and pick the lowest.
2. **Closed form:** set the derivative to zero, $k^* = \dfrac{\text{mean}\big(d\,(y - 0.5)\big)}{\text{mean}(d^2)}$ with $d$ = seed difference (this is least squares).
3. **Backtest:** choose $k$ on the past (≤ 2014) and score it on the future (≥ 2015). Choosing and scoring on the same games is like grading yourself with the answer key; it is always optimistic.

In [ ]:
# Method 1: grid search
for k in [0.00, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06]:
    p = 0.5 + k * tc.seed_diff
    print(k, round(((p - tc.y) ** 2).mean(), 4))

# Method 2: vertex of the parabola
d = tc.seed_diff
print('k_best =', round((d * (tc.y - 0.5)).mean() / (d ** 2).mean(), 4))

In [ ]:
# Method 3: choose k on <= 2014, score on >= 2015
train = tc[tc.Season <= 2014]
test  = tc[tc.Season >= 2015]

d = train.seed_diff
k_best = (d * (train.y - 0.5)).mean() / (d ** 2).mean()
p = 0.5 + k_best * test.seed_diff
print('k =', round(k_best, 4), '  test Brier =', round(((p - test.y) ** 2).mean(), 4))

# For comparison: "cheating" by choosing k on the test set itself
d = test.seed_diff
k_cheat = (d * (test.y - 0.5)).mean() / (d ** 2).mean()
p = 0.5 + k_cheat * test.seed_diff
print('cheating k =', round(k_cheat, 4), '  Brier =', round(((p - test.y) ** 2).mean(), 4))

**Findings.** The best $k$ is about 0.033: each seed line is worth about 3.3 percentage points of win probability, and this barely changes over decades. The honest test score is **0.196**. This is the baseline every later idea has to beat.

## 5. Season strength rating (Massey)

Seeds have only 16 levels and are chosen by people. We can build a finer, data-driven rating from every regular-season score.

Each game gives one equation:

$$\text{margin} = r_{\text{winner}} - r_{\text{loser}} + h \times \text{location}$$

with location = +1 if the winner was at home, −1 if away, 0 if neutral. Thousands of these equations contradict each other (games are noisy), so we solve them all at once by **least squares**: the ratings that make the squared errors as small as possible. $h$ is the home-court advantage, estimated together with the ratings.

An equivalent way to see it: **rating = average margin + average rating of the opponents**. Beating weak teams by a lot is discounted automatically. This matters because teams play only ~30 of ~360 possible opponents, mostly inside their own conference, so schedules differ hugely in difficulty.

In [ ]:
g = rs[rs.Season == 2025]
team_list = np.unique(np.r_[g.WTeamID, g.LTeamID])
idx = {t: i for i, t in enumerate(team_list)}

n = len(g)
X = np.zeros((n, len(team_list) + 1))            # one row per game, one column per team (+ home column)
X[np.arange(n), g.WTeamID.map(idx)] = 1
X[np.arange(n), g.LTeamID.map(idx)] = -1
X[:, -1] = g.WLoc.map({'H': 1, 'A': -1, 'N': 0})

margin = (g.WScore - g.LScore).values
coef, *_ = np.linalg.lstsq(X, margin, rcond=None)
rating = pd.DataFrame({'TeamID': team_list, 'rating': coef[:-1]})

print('Home-court advantage (points):', round(coef[-1], 2))
print('Game-to-game noise (residual std):', round((margin - X @ coef).std(), 2))
rating.merge(teams, on='TeamID').sort_values('rating', ascending=False).head(10)

**Findings.** Home court is worth about **3 points**, matching the usual rule of thumb, and the model was never told this. The top four teams (Duke, Auburn, Houston, Florida) are exactly the four 2025 No. 1 seeds. A rating difference is the expected margin on a neutral court.

The cell below solves the same system a different way: start everyone at 0 and repeatedly apply "average margin + average opponent rating" until nothing changes. The ranking matches; values are about 2 points higher for top teams because this version does not remove home-court advantage, and strong teams play more home games.

In [ ]:
d = pd.concat([
    pd.DataFrame({'t': g.WTeamID, 'opp': g.LTeamID, 'm': g.WScore - g.LScore}),
    pd.DataFrame({'t': g.LTeamID, 'opp': g.WTeamID, 'm': g.LScore - g.WScore}),
])
avg_margin = d.groupby('t').m.mean()

r = pd.Series(0.0, index=avg_margin.index)
for it in range(300):
    opp_avg = d.assign(opp_r=d.opp.map(r)).groupby('t').opp_r.mean()
    r = avg_margin + opp_avg
    r = r - r.mean()                              # fix the average at 0

r.sort_values(ascending=False).head(5)

### 5a. Ratings for every season, attached to tournament games

Ratings use **regular-season games only**, so no tournament result leaks into the features.

In [ ]:
def massey(g):
    team_list = np.unique(np.r_[g.WTeamID, g.LTeamID])
    idx = {t: i for i, t in enumerate(team_list)}
    n = len(g)
    X = np.zeros((n, len(team_list) + 1))
    X[np.arange(n), g.WTeamID.map(idx)] = 1
    X[np.arange(n), g.LTeamID.map(idx)] = -1
    X[:, -1] = g.WLoc.map({'H': 1, 'A': -1, 'N': 0})
    coef, *_ = np.linalg.lstsq(X, (g.WScore - g.LScore).values, rcond=None)
    return pd.DataFrame({'TeamID': team_list, 'rating': coef[:-1]})

ratings = pd.concat([massey(rs[rs.Season == s]).assign(Season=s)
                     for s in sorted(rs.Season.unique())])
print('Ratings computed:', ratings.shape)

tc = tc.drop(columns=['rA', 'rB', 'r_diff'], errors='ignore')     # safe to re-run
tc = tc.merge(ratings.rename(columns={'TeamID': 'A', 'rating': 'rA'}), on=['Season', 'A'])
tc = tc.merge(ratings.rename(columns={'TeamID': 'B', 'rating': 'rB'}), on=['Season', 'B'])
tc['r_diff'] = tc.rA - tc.rB
tc[['Season', 'A', 'B', 'y', 'seed_diff', 'r_diff']].tail()

### 5b. From rating difference to win probability (logistic regression)

A rating difference is an expected margin. The actual margin scatters around it in a bell curve, and the win probability is the area of that bell to the right of 0. As the difference grows, this traces an **S-shaped curve** that never goes below 0 or above 1. A straight line like the seed model would exceed 1 for large differences.

Logistic regression fits that S-curve, and can combine several inputs:

$$p = \frac{1}{1 + e^{-z}}, \qquad z = c_1 \cdot \text{seed\_diff} + c_2 \cdot \text{r\_diff} + b$$

The weights $c$ are found automatically by minimising the penalty on the training games (sliding downhill on a bowl-shaped error surface).

In [ ]:
from sklearn.linear_model import LogisticRegression

train = tc[tc.Season <= 2014]
test  = tc[tc.Season >= 2015]

for cols in [['seed_diff'], ['r_diff'], ['seed_diff', 'r_diff']]:
    m = LogisticRegression().fit(train[cols], train.y)
    p = m.predict_proba(test[cols])[:, 1]
    print(cols, ' Brier =', round(((p - test.y) ** 2).mean(), 4), ' weights =', m.coef_.round(3))

**Findings.** The rating beats seeds clearly (about 0.188 vs 0.197). Adding seeds on top does **not** help: a seed is roughly a rounded-off version of the rating, so it adds no new information, and the extra weight only adds noise. When both are used, the seed weight shrinks to about a third.

Manually tuning the two weights can lower the test score only very slightly, and only by "peeking" at the test set; the best weight found that way for seeds is zero.

## 6. Elo rating

Every team starts at 1500 and games are processed **in date order**:

1. Before the game, the expected win probability is $E_A = 1 / \big(1 + 10^{-(R_A - R_B)/400}\big)$ (another S-curve).
2. After the game, $R_A \leftarrow R_A + K\,(S - E_A)$, where $S$ is 1 for a win and 0 for a loss. B loses the same amount.

Beating a weak team earns little; losing to one costs a lot. Between seasons ratings regress 25% toward 1500 because rosters change.

| | Massey rating | Elo |
|---|---|---|
| How | solve all games at once | update game by game |
| Uses | point margin | win / loss only |
| Game weights | all equal | recent games matter more |
| Across seasons | starts fresh every season | carries last season over |

In [ ]:
def run_elo(rs, K=20, HA=75, carry=0.75):
    elo, snaps = {}, []
    for season, g in rs.sort_values(['Season', 'DayNum']).groupby('Season'):
        elo = {t: carry * r + (1 - carry) * 1500 for t, r in elo.items()}   # regress toward 1500
        for w, l, loc in zip(g.WTeamID, g.LTeamID, g.WLoc):
            rw, rl = elo.get(w, 1500), elo.get(l, 1500)
            ha = HA if loc == 'H' else (-HA if loc == 'A' else 0)          # home bonus of 75 Elo points
            e_w = 1 / (1 + 10 ** (-(rw + ha - rl) / 400))                   # winner's expected probability
            elo[w] = rw + K * (1 - e_w)
            elo[l] = rl - K * (1 - e_w)
        snaps.append(pd.DataFrame({'Season': season, 'TeamID': list(elo), 'elo': list(elo.values())}))
    return pd.concat(snaps)

elos = run_elo(rs)

tc = tc.drop(columns=['eA', 'eB', 'elo_diff'], errors='ignore')
tc = tc.merge(elos.rename(columns={'TeamID': 'A', 'elo': 'eA'}), on=['Season', 'A'])
tc = tc.merge(elos.rename(columns={'TeamID': 'B', 'elo': 'eB'}), on=['Season', 'B'])
tc['elo_diff'] = tc.eA - tc.eB

train = tc[tc.Season <= 2014]
test  = tc[tc.Season >= 2015]
for cols in [['seed_diff'], ['elo_diff'], ['r_diff'], ['r_diff', 'elo_diff']]:
    m = LogisticRegression().fit(train[cols], train.y)
    p = m.predict_proba(test[cols])[:, 1]
    print(cols, ' Brier =', round(((p - test.y) ** 2).mean(), 4))

**Findings.** Elo on its own is the *worst* feature (0.199, even worse than seeds), because it ignores margins. Yet rating + Elo beats rating alone (0.1863 vs 0.1879). Elo carries information the season rating lacks: last season's strength and recent form. Two measurements whose errors come from different sources partly cancel when combined.

**Lesson:** judge a feature by what it adds to the existing model, not by how it performs alone.

## 7. Per-possession efficiency

Fast-paced teams have more possessions, so their margins are naturally larger, and a points-based rating overrates them. Two teams that are equally good *per possession* can have different margins *per game*:

| | fast team | slow team |
|---|---|---|
| possessions per game | 80 | 60 |
| margin per game | +8 | +6 |
| margin per 100 possessions | +10 | +10 |

So we replace the margin with **margin per 100 possessions** and solve the same Massey system. Possessions are estimated from box scores: a possession ends with a shot, a turnover, or free throws, and an offensive rebound continues the same possession:

$$\text{possessions} \approx \text{FGA} - \text{OR} + \text{TO} + 0.475 \times \text{FTA}$$

Box scores exist from 2003 (men) and 2010 (women), so all models in this cell are compared on the same 2003+ games.

In [ ]:
det = pd.read_csv(D + 'MRegularSeasonDetailedResults.csv')

poss_W = det.WFGA - det.WOR + det.WTO + 0.475 * det.WFTA
poss_L = det.LFGA - det.LOR + det.LTO + 0.475 * det.LFTA
det['poss'] = (poss_W + poss_L) / 2
det['eff_margin'] = 100 * (det.WScore - det.LScore) / det.poss
print('Average possessions per game:', round(det.poss.mean(), 1))

def massey_eff(g):
    team_list = np.unique(np.r_[g.WTeamID, g.LTeamID])
    idx = {t: i for i, t in enumerate(team_list)}
    n = len(g)
    X = np.zeros((n, len(team_list) + 1))
    X[np.arange(n), g.WTeamID.map(idx)] = 1
    X[np.arange(n), g.LTeamID.map(idx)] = -1
    X[:, -1] = g.WLoc.map({'H': 1, 'A': -1, 'N': 0})
    coef, *_ = np.linalg.lstsq(X, g.eff_margin.values, rcond=None)
    return pd.DataFrame({'TeamID': team_list, 'eff': coef[:-1]})

effs = pd.concat([massey_eff(det[det.Season == s]).assign(Season=s)
                  for s in sorted(det.Season.unique())])

tc2 = tc.merge(effs.rename(columns={'TeamID': 'A', 'eff': 'fA'}), on=['Season', 'A'])
tc2 = tc2.merge(effs.rename(columns={'TeamID': 'B', 'eff': 'fB'}), on=['Season', 'B'])
tc2['eff_diff'] = tc2.fA - tc2.fB

train = tc2[tc2.Season <= 2014]
test  = tc2[tc2.Season >= 2015]
for cols in [['r_diff'], ['eff_diff'], ['r_diff', 'elo_diff'],
             ['eff_diff', 'elo_diff'], ['r_diff', 'eff_diff', 'elo_diff']]:
    m = LogisticRegression().fit(train[cols], train.y)
    p = m.predict_proba(test[cols])[:, 1]
    print(cols, ' Brier =', round(((p - test.y) ** 2).mean(), 4))

**Findings.** Efficiency beats the points rating (0.1873 vs 0.1883), and once efficiency is in, the points rating adds nothing. Decision: **replace** the points rating with efficiency. The best men's model so far is **efficiency + Elo (0.1861)**.

## 8. Expert rankings (Massey Ordinals, men only)

`MMasseyOrdinals` holds weekly rankings from dozens of systems (KenPom, Sagarin, ...). We take each system's last ranking before the tournament and average the 10 longest-running systems.

**Ranks are not on a linear scale.** The gap between #1 and #10 is large; the gap between #200 and #210 is tiny, because mid-table teams are packed together. A standard community transform stretches the top and compresses the bottom:

$$\text{score} = 100 - 4\ln(\text{rank} + 1) - \text{rank}/22$$

| rank | score | gap |
|---|---|---|
| 1 → 10 | 97.2 → 90.0 | 7.2 |
| 200 → 210 | 69.7 → 69.1 | 0.6 |

In [ ]:
mo = pd.read_csv(D + 'MMasseyOrdinals.csv')                     # large file, takes a few seconds
mo = mo[mo.RankingDayNum <= 133]                                # rankings before the tournament
last = mo.groupby(['Season', 'SystemName']).RankingDayNum.transform('max')
mo = mo[mo.RankingDayNum == last]                               # each system's final ranking

systems = ['POM', 'MOR', 'WLK', 'DOL', 'COL', 'RTH', 'WOL', 'SAG', 'DUN', 'BIH']
mo = mo[mo.SystemName.isin(systems)].copy()
mo['score'] = 100 - 4 * np.log(mo.OrdinalRank + 1) - mo.OrdinalRank / 22
ords = mo.groupby(['Season', 'TeamID']).agg(score=('score', 'mean'),
                                            rank=('OrdinalRank', 'mean')).reset_index()

tc3 = tc2.merge(ords.rename(columns={'TeamID': 'A', 'score': 'oA', 'rank': 'rkA'}), on=['Season', 'A'])
tc3 = tc3.merge(ords.rename(columns={'TeamID': 'B', 'score': 'oB', 'rank': 'rkB'}), on=['Season', 'B'])
tc3['ord_diff']  = tc3.oA - tc3.oB          # transformed score difference
tc3['rank_diff'] = tc3.rkB - tc3.rkA        # raw rank difference (smaller rank = stronger)

train = tc3[tc3.Season <= 2014]
test  = tc3[tc3.Season >= 2015]
print('Games:', len(tc3))
for cols in [['rank_diff'], ['ord_diff'], ['eff_diff', 'elo_diff'], ['eff_diff', 'elo_diff', 'ord_diff']]:
    m = LogisticRegression().fit(train[cols], train.y)
    p = m.predict_proba(test[cols])[:, 1]
    print(cols, ' Brier =', round(((p - test.y) ** 2).mean(), 4))

**Findings.**
- The transform matters a lot: raw ranks score 0.1953, transformed scores 0.1886. Logistic regression can only fit "difference × weight", so a feature has to be on a scale where differences mean the same thing everywhere.
- Expert rankings add **nothing** on top of efficiency + Elo. Many of these systems (e.g. KenPom) are themselves built on per-possession efficiency, so they repeat what we already have.
- Decision: drop them. A bonus: men and women can now use exactly the same features (women have no ordinals).

## 9. Toolbox

Everything needed from here on, in one self-contained cell. After reopening Colab you only need to run **Section 1 + this cell**.

`build(g)` returns one row per tournament game for `g = 'M'` or `'W'`, with:
- `y`: did A win
- `seed_diff`, `eff_diff`, `elo_diff`: A minus B
- `home`: +1 if A played at home, −1 if away, 0 if neutral (women's early rounds are hosted by top seeds)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

D = 'data/'

def run_elo(rs, K=20, HA=75, carry=0.75):
    elo, snaps = {}, []
    for season, g in rs.sort_values(['Season', 'DayNum']).groupby('Season'):
        elo = {t: carry * r + (1 - carry) * 1500 for t, r in elo.items()}
        for w, l, loc in zip(g.WTeamID, g.LTeamID, g.WLoc):
            rw, rl = elo.get(w, 1500), elo.get(l, 1500)
            ha = HA if loc == 'H' else (-HA if loc == 'A' else 0)
            e_w = 1 / (1 + 10 ** (-(rw + ha - rl) / 400))
            elo[w] = rw + K * (1 - e_w)
            elo[l] = rl - K * (1 - e_w)
        snaps.append(pd.DataFrame({'Season': season, 'TeamID': list(elo), 'elo': list(elo.values())}))
    return pd.concat(snaps)

def massey_fit(g, target):
    team_list = np.unique(np.r_[g.WTeamID, g.LTeamID])
    idx = {t: i for i, t in enumerate(team_list)}
    n = len(g)
    X = np.zeros((n, len(team_list) + 1))
    X[np.arange(n), g.WTeamID.map(idx)] = 1
    X[np.arange(n), g.LTeamID.map(idx)] = -1
    X[:, -1] = g.WLoc.map({'H': 1, 'A': -1, 'N': 0})
    coef, *_ = np.linalg.lstsq(X, g[target].values, rcond=None)
    return pd.DataFrame({'TeamID': team_list, 'val': coef[:-1]})

def efficiency_ratings(det):
    poss = ((det.WFGA - det.WOR + det.WTO + 0.475 * det.WFTA) +
            (det.LFGA - det.LOR + det.LTO + 0.475 * det.LFTA)) / 2
    det = det.assign(eff_margin=100 * (det.WScore - det.LScore) / poss)
    return pd.concat([massey_fit(det[det.Season == s], 'eff_margin').assign(Season=s)
                      for s in sorted(det.Season.unique())])

def add_diff(df, table, col, name):
    # attach a team-level value to A and B and store A - B
    t = table.rename(columns={col: 'v'})
    df = df.merge(t.rename(columns={'TeamID': 'A', 'v': 'vA'}), on=['Season', 'A'])
    df = df.merge(t.rename(columns={'TeamID': 'B', 'v': 'vB'}), on=['Season', 'B'])
    df[name] = df.vA - df.vB
    return df.drop(columns=['vA', 'vB'])

def build(g):
    rs    = pd.read_csv(D + f'{g}RegularSeasonCompactResults.csv')
    det   = pd.read_csv(D + f'{g}RegularSeasonDetailedResults.csv')
    tc    = pd.read_csv(D + f'{g}NCAATourneyCompactResults.csv')
    seeds = pd.read_csv(D + f'{g}NCAATourneySeeds.csv')
    seeds['seed'] = seeds.Seed.str[1:3].astype(int)

    tc['y'] = (tc.WTeamID < tc.LTeamID).astype(int)
    tc['A'] = tc[['WTeamID', 'LTeamID']].min(axis=1)
    tc['B'] = tc[['WTeamID', 'LTeamID']].max(axis=1)
    winner_home = tc.WLoc.map({'H': 1, 'A': -1, 'N': 0})
    tc['home'] = np.where(tc.y == 1, winner_home, -winner_home)

    tc = add_diff(tc, seeds[['Season', 'TeamID', 'seed']], 'seed', 'seed_diff')
    tc['seed_diff'] = -tc.seed_diff                     # smaller seed = stronger
    tc = add_diff(tc, efficiency_ratings(det), 'val', 'eff_diff')
    tc = add_diff(tc, run_elo(rs), 'elo', 'elo_diff')
    return tc

def sq(df):
    # per-game Brier penalty
    return (df.p - df.y) ** 2

def wf_preds(tc, cols):
    # walk-forward backtest: for each season S, train on all seasons before S and predict S
    out = []
    for S in range(2015, 2026):
        train, test = tc[tc.Season < S], tc[tc.Season == S]
        if len(test) == 0:                              # no tournament in 2020
            continue
        m = LogisticRegression().fit(train[cols], train.y)
        out.append(test[['Season', 'y']].assign(p=m.predict_proba(test[cols])[:, 1]))
    return pd.concat(out)

data = {g: build(g) for g in 'MW'}
print({g: len(t) for g, t in data.items()}, 'tournament games with features')

## 10. Walk-forward backtest (men)

A single split (≤ 2014 vs ≥ 2015) gives one number and cannot tell a real gain from a lucky year. Instead, for each season from 2015 to 2025 (no tournament in 2020) we train on **all earlier seasons** and predict that season. This gives 10 independent scores and mimics exactly what happens when we submit for 2026.

To compare two models fairly we count **in how many seasons one beats the other**, since every season's difficulty affects both models equally.

In [ ]:
sets_M = {
    'seed':             ['seed_diff'],
    'eff':              ['eff_diff'],
    'eff + Elo':        ['eff_diff', 'elo_diff'],
    'eff + Elo + seed': ['eff_diff', 'elo_diff', 'seed_diff'],
}
PM = {name: wf_preds(data['M'], cols) for name, cols in sets_M.items()}
RM = pd.DataFrame({name: sq(p).groupby(p.Season).mean() for name, p in PM.items()})

print(RM.round(4))
print('\nMean:\n', RM.mean().round(4))
print('\nStd across seasons:\n', RM.std().round(4))

d = RM['eff + Elo'] - RM['eff']
print(f'\neff + Elo beats eff in {(d < 0).sum()} / {len(d)} seasons, mean difference {d.mean():.4f}')

**Findings.**
- **Season-to-season swings dwarf model differences.** The same model scores 0.139 in 2025 and 0.219 in 2021, while models differ by 0.001–0.01 within a season. Averages alone can easily be fooled by luck.
- The worst seasons are known chaos years: 2018 (first 16-over-1 upset, UMBC over Virginia), 2021 (bubble tournament in Indiana), 2022 (15-seed Saint Peter's in the Elite Eight), 2023 (16-seed FDU over 1-seed Purdue; no 1-seed in the Elite Eight).
- In 2022 and 2023 our models are *worse than seeds alone*: they are more confident, and confident misses are punished hardest. Confidence is a double-edged sword.
- Efficiency + Elo beats efficiency alone in 7 of 10 seasons (mean −0.0012). A fair coin gives 7+ heads about 17% of the time, so the evidence is suggestive but not conclusive. We keep Elo because the direction is consistent and the reason is clear. Adding seeds makes things worse.

## 11. Model blending: logistic regression + LightGBM

Logistic regression draws a smooth S-curve along one straight direction. **LightGBM** adds up hundreds of small decision trees, each splitting the feature space into rectangles, so it can bend. Averaging two models whose mistakes differ can cancel some errors:

$$p = w \cdot p_{\text{LR}} + (1 - w) \cdot p_{\text{LightGBM}}$$

Two safeguards for LightGBM:
- **Monotone constraints:** a larger efficiency or Elo difference may never *lower* A's win probability.
- **Symmetry:** every training game is added twice, once as (A, B) and once flipped as (B, A) with negated features and outcome; predictions are averaged over both orientations so that P(A wins) + P(B wins) = 1.

In [ ]:
import lightgbm as lgb
import warnings; warnings.filterwarnings('ignore')

def flip(df, cols):
    f = df.copy()
    f[cols] = -f[cols]
    f['y'] = 1 - f.y
    return f

def walk_forward_blend(tc, cols, weights=(0, 0.3, 0.5, 0.7, 1.0)):
    rows = []
    for S in range(2015, 2026):
        train, test = tc[tc.Season < S], tc[tc.Season == S]
        if len(test) == 0:
            continue
        train2 = pd.concat([train, flip(train, cols)])

        lr = LogisticRegression().fit(train2[cols], train2.y)
        p_lr = lr.predict_proba(test[cols])[:, 1]

        gb = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.02, num_leaves=4,
                                min_child_samples=40, subsample=0.8, subsample_freq=1,
                                monotone_constraints=[1] * len(cols), verbose=-1)
        gb.fit(train2[cols], train2.y)
        p_gb = (gb.predict_proba(test[cols])[:, 1]
                + 1 - gb.predict_proba(flip(test, cols)[cols])[:, 1]) / 2

        row = {'Season': S}
        for w in weights:
            p = w * p_lr + (1 - w) * p_gb
            row[f'LR weight {w}'] = ((p - test.y) ** 2).mean()
        rows.append(row)
    return pd.DataFrame(rows).set_index('Season')

RB = walk_forward_blend(data['M'], ['eff_diff', 'elo_diff'])
print(RB.round(4))
print('\nMean:\n', RB.mean().round(4))
d = RB['LR weight 0.5'] - RB['LR weight 1.0']
print(f'\n50/50 blend vs LR only: wins {(d < 0).sum()} / {len(d)} seasons, mean difference {d.mean():.4f}')

**Findings.** LightGBM alone is worse (0.1872), and blending does not beat plain logistic regression (0.1862 vs 0.1861). With only two features, the relationship is essentially a smooth S-curve, so the extra flexibility of trees has nothing to find and only picks up noise. **Decision: keep logistic regression.**

## 12. Women, combined score, and confidence intervals

The same pipeline runs on the women's data by switching the file prefix to `W`. Differences:
- Women's games are more one-sided (average tournament margin ≈ 16.7 vs 11.8), so they are easier to predict and get their own model with a steeper S-curve.
- Women's early rounds are **hosted by top seeds**, so the `home` feature is tested.
- Box scores start in 2010 and there are no expert rankings.

Kaggle scores **all men's and women's games together**, so we also compute the combined score over all 1,315 test games.

Finally, a **95% bootstrap interval** for each difference: resample the games with replacement 2,000 times and look at the spread of the mean difference. If the interval crosses 0, the difference could be luck.

In [ ]:
sets_W = {
    'seed':             ['seed_diff'],
    'eff':              ['eff_diff'],
    'eff + Elo':        ['eff_diff', 'elo_diff'],
    'eff + home':       ['eff_diff', 'home'],
    'eff + Elo + home': ['eff_diff', 'elo_diff', 'home'],
}
PW = {name: wf_preds(data['W'], cols) for name, cols in sets_W.items()}
RW = pd.DataFrame({name: sq(p).groupby(p.Season).mean() for name, p in PW.items()})
print('Women, Brier by season:\n', RW.round(4))
print('\nWomen, mean:\n', RW.mean().round(4))

# Combined men + women, averaged over all games (as Kaggle does)
for name in ['seed', 'eff', 'eff + Elo']:
    men = PM['seed'] if name == 'seed' else PM['eff + Elo']
    allp = pd.concat([men, PW[name]])
    print(f'Combined (women = "{name}"): {len(allp)} games, Brier = {sq(allp).mean():.5f}')

# 95% bootstrap intervals for (variant - reference)
rng = np.random.default_rng(0)
def boot(a, b, n=2000):
    d = (sq(a) - sq(b)).values
    idx = rng.integers(0, len(d), (n, len(d)))
    means = d[idx].mean(axis=1)
    return round(d.mean(), 4), round(np.percentile(means, 2.5), 4), round(np.percentile(means, 97.5), 4)

print('\n(mean difference, 2.5%, 97.5%)')
print('Women  eff + Elo  - eff :', boot(PW['eff + Elo'], PW['eff']))
print('Women  eff + home - eff :', boot(PW['eff + home'], PW['eff']))
print('Men    eff + Elo  - eff :', boot(PM['eff + Elo'], PM['eff']))

**Findings.**
- Women: **efficiency alone is best (0.1384)**. Elo and home court do not help. Hosting is decided by seed, so "who plays at home" is already contained in team strength, and the women's history is short (the first backtest fold has only 315 training games).
- Combined score **0.1628**, essentially identical to a comparable published project (0.1630 on the same 1,315 games).
- All three intervals cross 0, including men's Elo. With ~650 test games per side, differences smaller than about ±0.0025 cannot be confirmed. Most remaining tweaks (blending, tuning) are below that size, so heavy tuning would mostly be fitting noise. When evidence is weak, prefer the simpler model unless there is a clear reason.

### Final model
| | Features | Backtest Brier |
|---|---|---|
| Men | efficiency + Elo | 0.1864 |
| Women | efficiency | 0.1384 |
| Combined | | **0.1628** |

## 13. Submission for 2026

Two differences from the backtest:
1. **Train on all history** (every tournament up to 2025).
2. **Features for every Division I team**, not just tournament teams, because the submission lists all 132,133 possible matchups. Only the games that are actually played are scored.

In [ ]:
def team_features(g):
    # efficiency rating and Elo of every team in the latest season (2026)
    rs  = pd.read_csv(D + f'{g}RegularSeasonCompactResults.csv')
    det = pd.read_csv(D + f'{g}RegularSeasonDetailedResults.csv')
    s = det.Season.max()
    eff = efficiency_ratings(det[det.Season == s]).rename(columns={'val': 'eff'})[['TeamID', 'eff']]
    elo = run_elo(rs)
    elo = elo[elo.Season == s][['TeamID', 'elo']]
    return eff.merge(elo, on='TeamID', how='left')

FEATURES = {'M': ['eff_diff', 'elo_diff'], 'W': ['eff_diff']}

sub = pd.read_csv(D + 'SampleSubmissionStage2.csv')
parts = sub.ID.str.split('_', expand=True).astype(int)
sub['A'], sub['B'] = parts[1], parts[2]
sub['g'] = np.where(sub.A < 3000, 'M', 'W')                    # men 1xxx, women 3xxx

for g in 'MW':
    cols = FEATURES[g]
    model = LogisticRegression().fit(data[g][cols], data[g].y)  # all tournaments up to 2025

    tf = team_features(g)
    rows = sub[sub.g == g][['A', 'B']]
    a = rows.merge(tf.rename(columns={'TeamID': 'A', 'eff': 'effA', 'elo': 'eloA'}), on='A', how='left')
    b = rows.merge(tf.rename(columns={'TeamID': 'B', 'eff': 'effB', 'elo': 'eloB'}), on='B', how='left')
    X = pd.DataFrame({'eff_diff': a.effA.values - b.effB.values,
                      'elo_diff': a.eloA.values - b.eloB.values})
    print(g, 'missing values:', X[cols].isna().sum().to_dict(), '  weights:', model.coef_.round(3))
    sub.loc[sub.g == g, 'Pred'] = model.predict_proba(X[cols].fillna(0))[:, 1]

print('\nRows:', len(sub), '(expected 132133)')
print(sub.groupby('g').Pred.describe().round(3))
print('\nDuke (1181) vs Michigan (1276):', sub.loc[sub.ID == '2026_1181_1276', 'Pred'].round(3).values)

In [ ]:
sub[['ID', 'Pred']].to_csv('submission.csv', index=False)
try:
    from google.colab import files
    files.download('submission.csv')
except ImportError:
    print('Saved submission.csv')

**Submitting.** On the competition page, click **Late Submission** and upload `submission.csv`. Expect the real 2026 score to differ from the 0.163 backtest average in either direction; single seasons ranged from about 0.14 to 0.22 for men.

## 14. Lessons and possible next steps

**What worked:** a good strength rating (per-possession efficiency, opponent-adjusted by least squares) plus Elo for last season and recent form. Most of the gain came from features, not from model complexity.

**What did not:** seeds on top of ratings, expert rankings on top of efficiency, women's home court, and LightGBM blending. Each repeated information the model already had.

**Ideas not yet tried:**
- *Volatility:* teams that rely on three-pointers have wider outcome distributions; their probabilities should sit closer to 0.5.
- *Coach tournament experience* (`MTeamCoaches`).
- *Travel distance* to tournament venues (`MGameCities`).
- *Shrinking predictions toward 0.5* to reduce the damage in upset-heavy seasons.